<a href="https://colab.research.google.com/github/jianna4/Machine_learning/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-11-29 11:41:34--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.22’

book-crossings.zip. 100%[===================>]  24.88M   111MB/s    in 0.2s    

2025-11-29 11:41:34 (111 MB/s) - ‘book-crossings.zip.22’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code

In [ ]:
# Step 1: Filter users and books
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

# Keep users with >=200 ratings, books with >=100 ratings
filtered_users = user_counts[user_counts >= 200].index
filtered_books = book_counts[book_counts >= 100].index

# Apply filters
df_ratings_filtered = df_ratings[
    df_ratings['user'].isin(filtered_users) &
    df_ratings['isbn'].isin(filtered_books)
]

# Merge with books
df = df_ratings_filtered.merge(df_books, on='isbn')

# Step 2: Create pivot table (user x book)
# IMPORTANT: Use 'title' as column, not 'isbn' — BUT deduplicate first!
# Because multiple ISBNs can map to same title, we group by title and take mean rating
df_grouped = df.groupby(['user', 'title'])['rating'].mean().reset_index()
rating_pivot = df_grouped.pivot_table(
    index='user',
    columns='title',
    values='rating',
    fill_value=0
)

# Step 3: Prepare for KNN
rating_matrix = csr_matrix(rating_pivot.values)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6, n_jobs=-1)
model_knn.fit(rating_matrix.T)  # books are columns → transpose

# Book titles in order
book_titles = rating_pivot.columns.tolist()
title_to_idx = {title: idx for idx, title in enumerate(book_titles)}

# Step 4: Define get_recommends
def get_recommends(book_title):
    if book_title not in title_to_idx:
        raise ValueError(f"Book '{book_title}' not found or filtered out.")

    idx = title_to_idx[book_title]
    distances, indices = model_knn.kneighbors(
        rating_matrix.T[idx].reshape(1, -1),
        n_neighbors=6
    )

    recommendations = []
    for i in range(1, 6):
        rec_title = book_titles[indices[0][i]]
        rec_dist = float(distances[0][i])
        recommendations.append([rec_title, rec_dist])
     # Sort recommendations by distance in descending order
    recommendations.sort(key=lambda x: x[1], reverse=True)
    N
    return [book_title, recommendations]

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]
You passed the challenge! 🎉🎉🎉🎉🎉
